In [1]:
import pandas as pd
from datetime import datetime
import glob
import warnings

def generar_informes_y_exportar(ruta_archivo, intervalo_meses=3):
    # Detección de separador
    with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
        primera_linea = f.readline()
        separador_real = ';' if primera_linea.count(';') > primera_linea.count(',') else ','

    try:
        df = pd.read_csv(ruta_archivo, sep=separador_real, on_bad_lines='skip', low_memory=False, dtype=str)
    except Exception as e:
        print(f"Error crítico al leer el archivo: {e}")
        return

    # Normalización de cabeceras
    df.columns = (df.columns.str.strip()
                  .str.upper()
                  .str.replace('Ó', 'O', regex=False)
                  .str.replace('Í', 'I', regex=False)
                  .str.replace('Á', 'A', regex=False)
                  .str.replace('É', 'E', regex=False)
                  .str.replace('Ú', 'U', regex=False)
                  .str.replace('  ', ' ', regex=False))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df['FECHA VALORACION NURICIONAL'] = pd.to_datetime(
            df['FECHA VALORACION NURICIONAL'],
            errors='coerce',
            format='mixed',
            dayfirst=True
        )

    fecha_actual = pd.to_datetime(datetime.now())
    contratos = df['NUMERO CONTRATO'].dropna().unique()

    for contrato in contratos:
        print("="*80)
        print(f"Procesando contrato: {contrato}")
        print("="*80)

        df_contrato = df[df['NUMERO CONTRATO'] == contrato]

        # DataFrame ordenado para extraer metadatos (como el ESTADO) siempre de la última toma
        df_ordenado = df_contrato.dropna(subset=['FECHA VALORACION NURICIONAL']).sort_values(['NUMERO DOCUMENTO BENEFICIARIO', 'FECHA VALORACION NURICIONAL'])
        df_ultimo_estado = df_ordenado.drop_duplicates(subset=['NUMERO DOCUMENTO BENEFICIARIO'], keep='last')

        # --- 1. Informe Unidades ---
        informe_unidades = df_contrato.groupby('NOMBRE UNIDAD')['NUMERO DOCUMENTO BENEFICIARIO'].nunique().reset_index()
        informe_unidades.rename(columns={'NUMERO DOCUMENTO BENEFICIARIO': 'TOTAL USUARIOS UNICOS'}, inplace=True)

        # --- 2. Tomas Faltantes ---
        df_fechas = df_ordenado.copy()
        usuarios_faltantes_ids = set()

        for usuario, grupo in df_fechas.groupby('NUMERO DOCUMENTO BENEFICIARIO'):
            fechas = grupo['FECHA VALORACION NURICIONAL'].dt.to_period('M').unique()
            if len(fechas) > 1:
                for i in range(1, len(fechas)):
                    diff_meses = (fechas[i].year - fechas[i-1].year) * 12 + (fechas[i].month - fechas[i-1].month)
                    if diff_meses > intervalo_meses:
                        usuarios_faltantes_ids.add(usuario)
                        break
            if usuario not in usuarios_faltantes_ids:
                ultima_toma = fechas[-1]
                diff_actual = (fecha_actual.year - ultima_toma.year) * 12 + (fecha_actual.month - ultima_toma.month)
                if diff_actual >= intervalo_meses:
                    usuarios_faltantes_ids.add(usuario)

        if usuarios_faltantes_ids:
            df_faltantes_detalle = df_ultimo_estado[df_ultimo_estado['NUMERO DOCUMENTO BENEFICIARIO'].isin(usuarios_faltantes_ids)][
                ['NOMBRE UNIDAD', 'NUMERO DOCUMENTO BENEFICIARIO', 'PRIMER NOMBRE BENEFICIARIO', 'PRIMER APELLIDO BENEFICIARIO', 'ESTADO']
            ]
        else:
            df_faltantes_detalle = pd.DataFrame(columns=['NOMBRE UNIDAD', 'NUMERO DOCUMENTO BENEFICIARIO', 'PRIMER NOMBRE BENEFICIARIO', 'PRIMER APELLIDO BENEFICIARIO', 'ESTADO'])

        print(f"\n--- 2. Usuarios con tomas faltantes (Total: {len(usuarios_faltantes_ids)}) ---")
        if not df_faltantes_detalle.empty:
            print(df_faltantes_detalle.to_string(index=False))
        else:
            print("Ningún usuario con tomas faltantes.")

        # --- 3. Desnutrición ---
        condicion_desnutricion = df_ultimo_estado['ESTADO PESO TALLA'].str.contains(r'desnutrici[oó]n', case=False, na=False, regex=True)

        df_desnutricion = df_ultimo_estado[condicion_desnutricion][
            ['NOMBRE UNIDAD', 'NUMERO DOCUMENTO BENEFICIARIO', 'PRIMER NOMBRE BENEFICIARIO', 'PRIMER APELLIDO BENEFICIARIO', 'ESTADO PESO TALLA', 'ESTADO', 'FECHA VALORACION NURICIONAL']
        ]

        df_desnutricion['FECHA VALORACION NURICIONAL'] = df_desnutricion['FECHA VALORACION NURICIONAL'].dt.strftime('%Y-%m-%d')

        # Generar archivo Excel
        nombre_excel = f'/content/Informe_Contrato_{contrato}.xlsx'
        with pd.ExcelWriter(nombre_excel, engine='openpyxl') as writer:
            informe_unidades.to_excel(writer, sheet_name='Usuarios por Unidad', index=False)

            if not df_faltantes_detalle.empty:
                df_faltantes_detalle.to_excel(writer, sheet_name='Tomas Faltantes', index=False)
            else:
                pd.DataFrame({'MENSAJE': ['Sin tomas faltantes registradas']}).to_excel(writer, sheet_name='Tomas Faltantes', index=False)

            if not df_desnutricion.empty:
                df_desnutricion.to_excel(writer, sheet_name='Alerta Desnutricion', index=False)
            else:
                pd.DataFrame({'MENSAJE': ['Sin casos registrados en la ultima toma']}).to_excel(writer, sheet_name='Alerta Desnutricion', index=False)

        print(f"\n✅ Excel guardado: {nombre_excel}\n")

# --- Ejecución ---
print("Buscando el archivo automáticamente en Colab...")
archivos_csv = glob.glob('/content/*ICBFCUEGeneralPorToma*.csv')

if not archivos_csv:
    print("❌ ERROR: No se encontró el archivo CSV.")
else:
    ruta_automatica = archivos_csv[0]
    generar_informes_y_exportar(ruta_automatica, intervalo_meses=3)

Buscando el archivo automáticamente en Colab...
Procesando contrato: 91002772026

--- 2. Usuarios con tomas faltantes (Total: 0) ---
Ningún usuario con tomas faltantes.

✅ Excel guardado: /content/Informe_Contrato_91002772026.xlsx



In [4]:
"""
Script unificado ICBF - Genera informes de tomas faltantes y alertas nutricionales
Soporta: ICBFCUEGeneralPorToma y GestanteLactantePorToma
Filtro automático: solo procesa beneficiarios con Estado = VINCULADO
"""

import pandas as pd
from datetime import datetime
import glob
import warnings
import os

# ──────────────────────────────────────────────
# CONFIGURACIÓN POR TIPO DE LIBRO
# ──────────────────────────────────────────────
PERFIL = {
    "general": {
        "col_diagnostico": "ESTADO PESO TALLA",
        "patron_alerta": r"desnutrici[oó]n",
        "hoja_alerta": "Alerta Desnutricion",
        "msg_sin_alerta": "Sin casos de desnutricion en la ultima toma",
    },
    "gestante": {
        "col_diagnostico": "EST.NUTR. GESTANTE",
        "patron_alerta": r"bajo peso|obesidad|sobrepeso",
        "hoja_alerta": "Alerta Nutricional",
        "msg_sin_alerta": "Sin alertas nutricionales en la ultima toma",
    },
}

# ──────────────────────────────────────────────
# NORMALIZACIÓN DE CABECERAS
# ──────────────────────────────────────────────
REEMPLAZOS = str.maketrans("ÓÍÁÉÚ\n", "OIAEU ")

def normalizar_col(col):
    return col.strip().upper().translate(REEMPLAZOS).replace("  ", " ").strip()


# ──────────────────────────────────────────────
# FUNCIÓN PRINCIPAL
# ──────────────────────────────────────────────
def generar_informes(ruta_archivo, tipo="general", intervalo_meses=3, carpeta_salida="/content"):
    ext = os.path.splitext(ruta_archivo)[1].lower()

    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(ruta_archivo, dtype=str)
    else:
        with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
            sep = ";" if f.readline().count(";") > f.readline().count(",") else ","
        try:
            df = pd.read_csv(ruta_archivo, sep=sep, on_bad_lines="skip", low_memory=False, dtype=str)
        except Exception as e:
            print(f"Error al leer el archivo: {e}")
            return

    df.columns = [normalizar_col(c) for c in df.columns]

    perfil = PERFIL[tipo]
    COL_DIAGNOSTICO = perfil["col_diagnostico"]
    COL_FECHA = "FECHA VALORACION NURICIONAL" if tipo == "general" else "FECHA VALORACION NUTRICIONAL"
    COL_ESTADO = "ESTADO"
    COL_CONTRATO = "NUMERO CONTRATO"
    COL_UNIDAD = "NOMBRE UNIDAD"
    COL_DOC = "NUMERO DOCUMENTO BENEFICIARIO"
    COL_NOMBRE = "PRIMER NOMBRE BENEFICIARIO"
    COL_APELLIDO = "PRIMER APELLIDO BENEFICIARIO"

    # Verificar columnas críticas
    for col in [COL_FECHA, COL_ESTADO, COL_CONTRATO, COL_DOC]:
        if col not in df.columns:
            print(f"⚠️  Columna no encontrada: '{col}'\nColumnas disponibles: {list(df.columns)}")
            return

    # ── Filtro automático: solo VINCULADOS ──
    df[COL_ESTADO] = df[COL_ESTADO].str.strip().str.upper()
    df = df[df[COL_ESTADO] == "VINCULADO"].copy()
    print(f"✅ Filtro aplicado: {len(df)} registros VINCULADOS procesados")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors="coerce", format="mixed", dayfirst=True)

    fecha_actual = pd.to_datetime(datetime.now())
    contratos = df[COL_CONTRATO].dropna().unique()

    os.makedirs(carpeta_salida, exist_ok=True)

    for contrato in contratos:
        print(f"\n{'='*70}\nProcesando contrato: {contrato}\n{'='*70}")

        df_c = df[df[COL_CONTRATO] == contrato]

        df_ord = (
            df_c.dropna(subset=[COL_FECHA])
            .sort_values([COL_DOC, COL_FECHA])
        )

        # Último registro por beneficiario (para diagnóstico y estado)
        df_ultimo = df_ord.drop_duplicates(subset=[COL_DOC], keep="last")

        # ── Hoja 1: Usuarios por unidad ──
        hoja_unidades = (
            df_c.groupby(COL_UNIDAD)[COL_DOC]
            .nunique()
            .reset_index()
            .rename(columns={COL_DOC: "TOTAL USUARIOS UNICOS"})
        )

        # ── Hoja 2: Tomas faltantes ──
        ids_faltantes = set()
        for usuario, grupo in df_ord.groupby(COL_DOC):
            fechas = grupo[COL_FECHA].dt.to_period("M").unique()
            if len(fechas) > 1:
                for i in range(1, len(fechas)):
                    diff = (fechas[i].year - fechas[i - 1].year) * 12 + (fechas[i].month - fechas[i - 1].month)
                    if diff > intervalo_meses:
                        ids_faltantes.add(usuario)
                        break
            if usuario not in ids_faltantes:
                ultima = fechas[-1]
                diff_hoy = (fecha_actual.year - ultima.year) * 12 + (fecha_actual.month - ultima.month)
                if diff_hoy >= intervalo_meses:
                    ids_faltantes.add(usuario)

        cols_faltantes = [COL_UNIDAD, COL_DOC, COL_NOMBRE, COL_APELLIDO]
        # Añadir columna diagnóstico si existe
        if COL_DIAGNOSTICO in df_ultimo.columns:
            cols_faltantes.append(COL_DIAGNOSTICO)

        df_faltantes = (
            df_ultimo[df_ultimo[COL_DOC].isin(ids_faltantes)][cols_faltantes]
            if ids_faltantes
            else pd.DataFrame(columns=cols_faltantes)
        )

        print(f"Tomas faltantes: {len(ids_faltantes)} usuarios")

        # ── Hoja 3: Alerta nutricional ──
        if COL_DIAGNOSTICO in df_ultimo.columns:
            condicion = df_ultimo[COL_DIAGNOSTICO].str.contains(
                perfil["patron_alerta"], case=False, na=False, regex=True
            )
            cols_alerta = [COL_UNIDAD, COL_DOC, COL_NOMBRE, COL_APELLIDO, COL_DIAGNOSTICO, COL_FECHA]
            df_alerta = df_ultimo[condicion][cols_alerta].copy()
            df_alerta[COL_FECHA] = df_alerta[COL_FECHA].dt.strftime("%Y-%m-%d")
            print(f"Alertas nutricionales: {len(df_alerta)} usuarios")
        else:
            df_alerta = pd.DataFrame()
            print(f"⚠️  Columna diagnóstico '{COL_DIAGNOSTICO}' no encontrada — hoja de alertas omitida")

        # ── Exportar Excel ──
        nombre_excel = os.path.join(carpeta_salida, f"Informe_{tipo.upper()}_Contrato_{contrato}.xlsx")
        with pd.ExcelWriter(nombre_excel, engine="openpyxl") as writer:
            hoja_unidades.to_excel(writer, sheet_name="Usuarios por Unidad", index=False)

            if not df_faltantes.empty:
                df_faltantes.to_excel(writer, sheet_name="Tomas Faltantes", index=False)
            else:
                pd.DataFrame({"MENSAJE": ["Sin tomas faltantes registradas"]}).to_excel(
                    writer, sheet_name="Tomas Faltantes", index=False
                )

            if not df_alerta.empty:
                df_alerta.to_excel(writer, sheet_name=perfil["hoja_alerta"], index=False)
            else:
                pd.DataFrame({"MENSAJE": [perfil["msg_sin_alerta"]]}).to_excel(
                    writer, sheet_name=perfil["hoja_alerta"], index=False
                )

        print(f"✅ Excel guardado: {nombre_excel}")


# ──────────────────────────────────────────────
# EJECUCIÓN AUTOMÁTICA EN COLAB/LOCAL
# ──────────────────────────────────────────────
if __name__ == "__main__":
    CARPETA = "/content"

    # ── Libro General ──
    archivos_general = glob.glob(f"{CARPETA}/*ICBFCUEGeneralPorToma*")
    if archivos_general:
        print("\n📂 Procesando: ICBFCUEGeneralPorToma")
        generar_informes(archivos_general[0], tipo="general", intervalo_meses=3, carpeta_salida=CARPETA)
    else:
        print("❌ No se encontró archivo ICBFCUEGeneralPorToma")

    # ── Libro Gestante/Lactante ──
    archivos_gestante = glob.glob(f"{CARPETA}/*GestanteLactantePorToma*")
    if archivos_gestante:
        print("\n📂 Procesando: GestanteLactantePorToma")
        generar_informes(archivos_gestante[0], tipo="gestante", intervalo_meses=3, carpeta_salida=CARPETA)
    else:
        print("❌ No se encontró archivo GestanteLactantePorToma")


📂 Procesando: ICBFCUEGeneralPorToma
✅ Filtro aplicado: 1043 registros VINCULADOS procesados

Procesando contrato: 91002702026
Tomas faltantes: 180 usuarios
Alertas nutricionales: 10 usuarios
✅ Excel guardado: /content/Informe_GENERAL_Contrato_91002702026.xlsx

Procesando contrato: 91002682026
Tomas faltantes: 98 usuarios
Alertas nutricionales: 2 usuarios
✅ Excel guardado: /content/Informe_GENERAL_Contrato_91002682026.xlsx

Procesando contrato: 91002532026
Tomas faltantes: 143 usuarios
Alertas nutricionales: 2 usuarios
✅ Excel guardado: /content/Informe_GENERAL_Contrato_91002532026.xlsx

📂 Procesando: GestanteLactantePorToma
✅ Filtro aplicado: 281 registros VINCULADOS procesados

Procesando contrato: 91002532026
Tomas faltantes: 0 usuarios
Alertas nutricionales: 40 usuarios
✅ Excel guardado: /content/Informe_GESTANTE_Contrato_91002532026.xlsx

Procesando contrato: 91002702026
Tomas faltantes: 0 usuarios
Alertas nutricionales: 13 usuarios
✅ Excel guardado: /content/Informe_GESTANTE_Cont

In [7]:
"""
Script unificado ICBF v2 - Tomas faltantes + Alertas nutricionales
Mejora v2: Cruza con archivo de activos para detectar:
  - Usuarios vinculados con 0 tomas (invisible en versión anterior)
  - Usuarios con menos tomas de las esperadas según fecha de vinculación
Filtro automático: solo procesa beneficiarios con Estado = VINCULADO
"""

import pandas as pd
from datetime import datetime
import glob
import warnings
import os
import math

# ──────────────────────────────────────────────
# CONFIGURACIÓN POR TIPO DE LIBRO
# ──────────────────────────────────────────────
PERFIL = {
    "general": {
        "col_doc_nutricion":  "Numero Documento Beneficiario",
        "col_diagnostico":    "ESTADO PESO TALLA",
        "patron_alerta":      r"desnutrici[oó]n",
        "hoja_alerta":        "Alerta Desnutricion",
        "msg_sin_alerta":     "Sin casos de desnutricion en la ultima toma",
        "tipos_beneficiario": ["MENOR DE SEIS MESES", "NIÑO O NIÑA ENTRE 6 MESES Y 5 AÑOS Y 11 MESES"],
    },
    "gestante": {
        "col_doc_nutricion":  "Número documento beneficiario",
        "col_diagnostico":    "EST.NUTR. GESTANTE",
        "patron_alerta":      r"bajo peso|obesidad|sobrepeso",
        "hoja_alerta":        "Alerta Nutricional",
        "msg_sin_alerta":     "Sin alertas nutricionales en la ultima toma",
        "tipos_beneficiario": ["PERSONA GESTANTE"],
    },
}

REEMPLAZOS = str.maketrans("ÓÍÁÉÚ\n", "OIAEU ")

def norm(col):
    return col.strip().upper().translate(REEMPLAZOS).replace("  ", " ").strip()

def leer_excel_o_csv(ruta, preferir_hoja=None):
    ext = os.path.splitext(ruta)[1].lower()
    if ext in (".xlsx", ".xls"):
        hojas = pd.ExcelFile(ruta).sheet_names
        hoja = hojas[0]
        if preferir_hoja:
            for h in hojas:
                if preferir_hoja.lower() in h.lower():
                    hoja = h
                    break
        if len(hojas) > 1:
            print(f"  ℹ️  Archivo con {len(hojas)} hojas. Usando: '{hoja}'")
        return pd.read_excel(ruta, sheet_name=hoja, dtype=str)
    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
        sep = ";" if f.readline().count(";") > f.readline().count(",") else ","
    return pd.read_csv(ruta, sep=sep, on_bad_lines="skip", low_memory=False, dtype=str)

def tomas_esperadas(meses_vinculado, intervalo):
    """Cuántas tomas debería tener un usuario según su antigüedad."""
    return max(1, math.floor(meses_vinculado / intervalo))

# ──────────────────────────────────────────────
# FUNCIÓN PRINCIPAL
# ──────────────────────────────────────────────
def generar_informes(ruta_nutricion, ruta_activos, tipo="general",
                     intervalo_meses=3, carpeta_salida="/content"):

    perfil   = PERFIL[tipo]
    clave_hoja = "ICBFCUEGeneralPorToma" if tipo == "general" else "GestanteLactantePorToma"
    df_nut   = leer_excel_o_csv(ruta_nutricion, preferir_hoja=clave_hoja)
    df_act   = leer_excel_o_csv(ruta_activos)

    # ── Normalizar cabeceras nutrición ──
    df_nut.columns = [norm(c) for c in df_nut.columns]

    COL_DIAGNOSTICO = perfil["col_diagnostico"]
    COL_FECHA       = "FECHA VALORACION NURICIONAL" if tipo == "general" else "FECHA VALORACION NUTRICIONAL"
    COL_ESTADO      = "ESTADO"
    COL_CONTRATO    = "NUMERO CONTRATO"
    COL_UNIDAD      = "NOMBRE UNIDAD"
    COL_DOC_NUT     = norm(perfil["col_doc_nutricion"])
    COL_NOMBRE      = "PRIMER NOMBRE BENEFICIARIO"
    COL_APELLIDO    = "PRIMER APELLIDO BENEFICIARIO"

    for col in [COL_FECHA, COL_ESTADO, COL_CONTRATO, COL_DOC_NUT]:
        if col not in df_nut.columns:
            print(f"⚠️  Columna no encontrada en nutrición: '{col}'")
            print(f"    Disponibles: {list(df_nut.columns)}")
            return

    # ── Filtro VINCULADO en nutrición ──
    df_nut[COL_ESTADO] = df_nut[COL_ESTADO].str.strip().str.upper()
    df_nut = df_nut[df_nut[COL_ESTADO] == "VINCULADO"].copy()
    print(f"✅ Registros VINCULADOS en nutrición: {len(df_nut)}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df_nut[COL_FECHA] = pd.to_datetime(df_nut[COL_FECHA], errors="coerce", format="mixed", dayfirst=True)

    # ── Preparar activos filtrados por tipo ──
    tipos_upper = [t.upper() for t in perfil["tipos_beneficiario"]]
    df_act["_tipo_upper"] = df_act["Nombre Tipo de beneficiario"].str.strip().str.upper()
    df_act_tipo = df_act[df_act["_tipo_upper"].isin(tipos_upper)].copy()

    df_act_tipo["_fecha_vinc"] = pd.to_datetime(
        df_act_tipo["Fecha de vinculación del beneficiario"],
        errors="coerce", dayfirst=True
    )
    df_act_tipo["_doc"] = df_act_tipo["Documento del beneficiario"].str.strip()

    fecha_actual = pd.to_datetime(datetime.now())
    df_act_tipo["_meses_vinculado"] = (
        (fecha_actual.year  - df_act_tipo["_fecha_vinc"].dt.year)  * 12 +
        (fecha_actual.month - df_act_tipo["_fecha_vinc"].dt.month)
    )
    df_act_tipo["_tomas_esperadas"] = df_act_tipo["_meses_vinculado"].apply(
        lambda m: tomas_esperadas(m, intervalo_meses) if pd.notna(m) else 0
    )

    # Conteo real de tomas por usuario en nutrición
    conteo_tomas = df_nut.groupby(COL_DOC_NUT).size().reset_index(name="_tomas_reales")
    df_act_tipo = df_act_tipo.merge(
        conteo_tomas, left_on="_doc", right_on=COL_DOC_NUT, how="left"
    )
    df_act_tipo["_tomas_reales"] = df_act_tipo["_tomas_reales"].fillna(0).astype(int)

    # Último registro nutrición por beneficiario (para diagnóstico)
    # Desempate: fecha + número de toma para cuando dos registros tienen la misma fecha
    COL_NTOMA = next(
        (c for c in df_nut.columns if "NUMERO DE TOMA" in c or "TOMA DEL BENEFICIARIO" in c or "N DE TOMA" in c),
        None
    )
    df_nut_ord = df_nut.dropna(subset=[COL_FECHA]).copy()
    df_nut_ord["_fecha_dt"] = pd.to_datetime(df_nut_ord[COL_FECHA], errors="coerce")
    if COL_NTOMA:
        df_nut_ord["_ntoma_num"] = pd.to_numeric(df_nut_ord[COL_NTOMA], errors="coerce").fillna(0)
        df_nut_ord = df_nut_ord.sort_values([COL_DOC_NUT, "_fecha_dt", "_ntoma_num"])
    else:
        df_nut_ord = df_nut_ord.sort_values([COL_DOC_NUT, "_fecha_dt"])
    df_ultimo  = df_nut_ord.drop_duplicates(subset=[COL_DOC_NUT], keep="last")

    os.makedirs(carpeta_salida, exist_ok=True)
    contratos = df_nut[COL_CONTRATO].dropna().unique()

    for contrato in contratos:
        print(f"\n{'='*70}\nProcesando contrato: {contrato}\n{'='*70}")

        df_c       = df_nut[df_nut[COL_CONTRATO] == contrato]
        df_ult_c   = df_ultimo[df_ultimo[COL_CONTRATO] == contrato] if COL_CONTRATO in df_ultimo.columns else df_ultimo

        # Activos de este contrato
        df_act_c = df_act_tipo[
            df_act_tipo["Número del Contrato"].str.strip() == str(contrato).strip()
        ].copy() if "Número del Contrato" in df_act_tipo.columns else df_act_tipo.copy()

        # ── Hoja 1: Usuarios por unidad ──
        hoja_unidades = (
            df_c.groupby(COL_UNIDAD)[COL_DOC_NUT]
            .nunique().reset_index()
            .rename(columns={COL_DOC_NUT: "TOTAL USUARIOS UNICOS"})
        )

        # ── Hoja 2: Tomas faltantes (lógica mejorada) ──
        # Caso A: usuarios con 0 tomas (no aparecen en nutrición pero sí en activos)
        sin_toma = df_act_c[df_act_c["_tomas_reales"] == 0].copy()

        # Caso B: usuarios con menos tomas de las esperadas (incluyendo los de 1 toma)
        con_deficit = df_act_c[
            (df_act_c["_tomas_reales"] > 0) &
            (df_act_c["_tomas_reales"] < df_act_c["_tomas_esperadas"])
        ].copy()

        # Unir ambos casos
        faltantes_activos = pd.concat([sin_toma, con_deficit], ignore_index=True)

        # Construir detalle de faltantes
        filas_faltantes = []
        for _, row in faltantes_activos.iterrows():
            doc = row["_doc"]
            # Datos nutricionales si existen
            reg_nut = df_ult_c[df_ult_c[COL_DOC_NUT] == doc]
            unidad  = reg_nut[COL_UNIDAD].values[0] if not reg_nut.empty and COL_UNIDAD in reg_nut.columns else row.get("Nombre de la unidad de servicio", "")
            nombre  = reg_nut[COL_NOMBRE].values[0] if not reg_nut.empty and COL_NOMBRE in reg_nut.columns else row.get("Primer Nombre del beneficiario", "")
            apellido= reg_nut[COL_APELLIDO].values[0] if not reg_nut.empty and COL_APELLIDO in reg_nut.columns else row.get("Primer apellido del beneficiario", "")
            diag    = reg_nut[COL_DIAGNOSTICO].values[0] if not reg_nut.empty and COL_DIAGNOSTICO in reg_nut.columns else "SIN TOMA REGISTRADA"

            filas_faltantes.append({
                "UNIDAD":             unidad,
                "DOCUMENTO":          doc,
                "NOMBRE":             nombre,
                "APELLIDO":           apellido,
                "TOMAS REALIZADAS":   int(row["_tomas_reales"]),
                "TOMAS ESPERADAS":    int(row["_tomas_esperadas"]),
                "MESES VINCULADO":    int(row["_meses_vinculado"]) if pd.notna(row["_meses_vinculado"]) else 0,
                "ULTIMO DIAGNOSTICO": diag,
                "MOTIVO":             "Sin toma registrada" if row["_tomas_reales"] == 0 else "Tomas insuficientes",
            })

        df_faltantes = pd.DataFrame(filas_faltantes)
        print(f"Tomas faltantes: {len(df_faltantes)} usuarios "
              f"({len(sin_toma)} sin toma, {len(con_deficit)} con déficit)")

        # ── Hoja 3: Alerta nutricional ──
        if COL_DIAGNOSTICO in df_ult_c.columns:
            condicion = df_ult_c[COL_DIAGNOSTICO].str.contains(
                perfil["patron_alerta"], case=False, na=False, regex=True
            )
            cols_alerta = [COL_UNIDAD, COL_DOC_NUT, COL_NOMBRE, COL_APELLIDO,
                           COL_DIAGNOSTICO, COL_FECHA]
            df_alerta = df_ult_c[condicion][cols_alerta].copy()
            df_alerta[COL_FECHA] = df_alerta[COL_FECHA].dt.strftime("%Y-%m-%d")
            print(f"Alertas nutricionales: {len(df_alerta)} usuarios")
        else:
            df_alerta = pd.DataFrame()

        # ── Exportar Excel ──
        nombre_excel = os.path.join(
            carpeta_salida, f"Informe_{tipo.upper()}_Contrato_{contrato}.xlsx"
        )
        with pd.ExcelWriter(nombre_excel, engine="openpyxl") as writer:
            hoja_unidades.to_excel(writer, sheet_name="Usuarios por Unidad", index=False)

            if not df_faltantes.empty:
                df_faltantes.to_excel(writer, sheet_name="Tomas Faltantes", index=False)
            else:
                pd.DataFrame({"MENSAJE": ["Sin tomas faltantes registradas"]}).to_excel(
                    writer, sheet_name="Tomas Faltantes", index=False)

            if not df_alerta.empty:
                df_alerta.to_excel(writer, sheet_name=perfil["hoja_alerta"], index=False)
            else:
                pd.DataFrame({"MENSAJE": [perfil["msg_sin_alerta"]]}).to_excel(
                    writer, sheet_name=perfil["hoja_alerta"], index=False)

        print(f"✅ Excel guardado: {nombre_excel}")


# ──────────────────────────────────────────────
# EJECUCIÓN AUTOMÁTICA EN COLAB
# ──────────────────────────────────────────────
if __name__ == "__main__":
    CARPETA = "/content"

    archivos_activos = glob.glob(f"{CARPETA}/*BeneficiariosPIActivos*")
    if not archivos_activos:
        print("❌ No se encontró archivo de activos (ICBFCUEBeneficiariosPIActivosRegionalizado)")
        exit()
    ruta_activos = archivos_activos[0]
    print(f"📋 Archivo de activos: {ruta_activos}")

    archivos_general = glob.glob(f"{CARPETA}/*ICBFCUEGeneralPorToma*")
    if archivos_general:
        print("\n📂 Procesando: ICBFCUEGeneralPorToma")
        generar_informes(archivos_general[0], ruta_activos, tipo="general",
                         intervalo_meses=3, carpeta_salida=CARPETA)
    else:
        print("❌ No se encontró archivo ICBFCUEGeneralPorToma")

    archivos_gestante = glob.glob(f"{CARPETA}/*GestanteLactantePorToma*")
    if archivos_gestante:
        print("\n📂 Procesando: GestanteLactantePorToma")
        generar_informes(archivos_gestante[0], ruta_activos, tipo="gestante",
                         intervalo_meses=3, carpeta_salida=CARPETA)
    else:
        print("❌ No se encontró archivo GestanteLactantePorToma")

❌ No se encontró archivo de activos (ICBFCUEBeneficiariosPIActivosRegionalizado)


IndexError: list index out of range